<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/20-llms-in-practice/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 20: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import json
import math
import string

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_cosine_similarity_matrix(f):
    rng = np.random.default_rng(0)
    A, B = rng.normal(size=(4, 6)), rng.normal(size=(3, 6))
    expected = (A / np.linalg.norm(A, axis=1, keepdims=True)) @ (B / np.linalg.norm(B, axis=1, keepdims=True)).T
    with_zero = np.vstack([A[:2], np.zeros((1, 6))])
    _scenarios("cosine_similarity_matrix", [
        ("the example", lambda: f(np.array([[1.0, 0.0]]), np.array([[1.0, 1.0], [0.0, 2.0]])), np.array([[2 ** -0.5, 0.0]]), None),
        ("random vectors", lambda: f(A, B), expected, "divide each row by its own length"),
        ("lengths don't matter", lambda: f(A * 10, B * 0.1), expected, None),
        ("a row of zeros gives 0, not nan", lambda: f(with_zero, B)[2], np.zeros(3), "skip the division where the length is 0"),
        ("shape", lambda: f(A, B).shape, (4, 3), None),
    ])


def check_top_k_indices(f):
    rng = np.random.default_rng(1)
    scores = rng.random(50)
    ties = np.random.default_rng(5).choice([0.1, 0.2, 0.3], 300)
    _scenarios("top_k_indices", [
        ("the example", lambda: f([0.2, 0.9, 0.5, 0.9], 3), [1, 3, 2], "ties: the earlier position first"),
        ("top 5 of 50", lambda: f(scores, 5), [int(i) for i in np.argsort(-scores)[:5]], None),
        ("all equal", lambda: f([0.5, 0.5, 0.5, 0.5], 4), [0, 1, 2, 3], "use a stable sort"),
        ("many ties", lambda: f(ties, 40), [int(i) for i in np.argsort(-ties, kind="stable")[:40]], "use a stable sort"),
        ("k larger than the list", lambda: f([3.0, 1.0], 5), [0, 1], None),
        ("returns Python ints", lambda: sorted({type(i).__name__ for i in f(scores, 3)}), ["int"], None),
    ])


def check_chunk_words(f):
    words = " ".join(f"w{i}" for i in range(10))
    _scenarios("chunk_words", [
        ("the example", lambda: f("a b c d e f g", 3, 1), ["a b c", "c d e", "e f g"], None),
        ("no overlap", lambda: f(words, 4, 0), ["w0 w1 w2 w3", "w4 w5 w6 w7", "w8 w9"], None),
        ("overlap 2, a short last chunk", lambda: f(words, 4, 2), ["w0 w1 w2 w3", "w2 w3 w4 w5", "w4 w5 w6 w7", "w6 w7 w8 w9"],
         "stop once a chunk reaches the last word"),
        ("text shorter than one chunk", lambda: f("just three words", 5, 2), ["just three words"], None),
        ("extra whitespace", lambda: f("  a\n b\t c  ", 2, 0), ["a b", "c"], None),
        ("empty text", lambda: f("", 3, 1), [], None),
        ("overlap equal to size raises", lambda: f(words, 3, 3), Raises(ValueError), None),
        ("overlap larger than size raises", lambda: f(words, 3, 5), Raises(ValueError), "check 0 <= overlap < size"),
        ("negative overlap raises", lambda: f(words, 3, -1), Raises(ValueError), None),
    ])


def check_format_chatml(f):
    convo = [{"role": "system", "content": "Be brief."}, {"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello!"},
             {"role": "user", "content": "Two\nlines"}]
    expected = ("<|im_start|>system\nBe brief.<|im_end|>\n<|im_start|>user\nHi<|im_end|>\n"
                "<|im_start|>assistant\nHello!<|im_end|>\n<|im_start|>user\nTwo\nlines<|im_end|>\n")
    _scenarios("format_chatml", [
        ("the example", lambda: f([{"role": "user", "content": "Hi"}], add_generation_prompt=True),
         "<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n", None),
        ("a whole conversation", lambda: f(convo), expected, "role, new line, content, <|im_end|>, new line"),
        ("with the generation prompt", lambda: f(convo, add_generation_prompt=True), expected + "<|im_start|>assistant\n", None),
        ("no messages", lambda: f([]), "", None),
        ("an unknown role raises", lambda: f([{"role": "assistent", "content": "x"}]), Raises(ValueError), None),
    ])


def check_build_rag_prompt(f):
    head = "Answer the question using only the sources below. If they don't contain the answer, say \"I don't know\".\n\nSources:\n"
    chunks = ["Cards cost 2 pounds.", "Open 9 to 7.", "Wi-Fi is free."]
    two = len("[1] Cards cost 2 pounds.\n[2] Open 9 to 7.")

    def full(sources, q="How much is a card?"):
        return head + sources + f"\n\nQuestion: {q}\nAnswer:"

    _scenarios("build_rag_prompt", [
        ("three sources", lambda: f("How much is a card?", chunks), full("[1] Cards cost 2 pounds.\n[2] Open 9 to 7.\n[3] Wi-Fi is free."),
         "check every character of the template"),
        ("a limit that fits two", lambda: f("How much is a card?", chunks, max_chars=two), full("[1] Cards cost 2 pounds.\n[2] Open 9 to 7."),
         "the sources block, with its new lines, must be at most max_chars"),
        ("a limit exactly one character short", lambda: f("How much is a card?", chunks, max_chars=two - 1), full("[1] Cards cost 2 pounds."), None),
        ("stops at the first that doesn't fit", lambda: f("q", ["short", "a much longer chunk of text", "tiny"], max_chars=20),
         full("[1] short", q="q"), None),
        ("nothing fits", lambda: f("q", chunks, max_chars=5), full("(no sources)", q="q"), None),
        ("no chunks", lambda: f("q", []), full("(no sources)", q="q"), None),
    ])


def check_retrieval_metrics(f):
    results = [["a", "b", "c", "d"], ["e", "f", "g", "h"], ["i", "j", "k", "l"]]
    relevant = [{"a", "c"}, {"h"}, {"z"}]
    _scenarios("retrieval_metrics", [
        ("the example", lambda: f([["a", "b", "c"], ["d", "e", "f"]], [{"b"}, {"x"}], 2), {"recall@k": 0.5, "mrr": 0.25}, None),
        ("two relevant documents", lambda: f(results[:1], relevant[:1], 2), {"recall@k": 0.5, "mrr": 1.0},
         "recall divides by the number of relevant documents"),
        ("the first hit beyond k still counts for MRR", lambda: f(results[1:2], relevant[1:2], 2), {"recall@k": 0.0, "mrr": 0.25}, None),
        ("three queries", lambda: f(results, relevant, 3), {"recall@k": (1.0 + 0 + 0) / 3, "mrr": (1 + 0.25 + 0) / 3}, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f(results, relevant, 3).values()}), ["float"], None),
        ("mismatched lengths raise", lambda: f(results, relevant[:2], 3), Raises(ValueError), None),
    ])


def check_answer_f1(f):
    _scenarios("answer_f1", [
        ("the example", lambda: f("It costs 2 pounds.", "2 pounds"), 2 / 3, None),
        ("identical after normalising", lambda: f("The library card!", "library card"), 1.0, "lower case, remove punctuation and a/an/the"),
        ("no words in common", lambda: f("Saturday", "10:00"), 0.0, None),
        ("repeated words count once", lambda: f("no no no", "no"), 0.5, "use Counter(...) & Counter(...)"),
        ("repeats on both sides", lambda: f("no no", "no no maybe"), 0.8, "a word repeated in both counts as often as in the shorter"),
        ("partial overlap", lambda: f("open on Saturday mornings", "Saturday from 10 to 4"), 2 * (1 / 4) * (1 / 5) / (1 / 4 + 1 / 5), None),
        ("both empty after normalising", lambda: f("The.", "a"), 1.0, None),
        ("one empty", lambda: f("", "2 pounds"), 0.0, None),
    ])


def check_extract_json(f):
    _scenarios("extract_json", [
        ("plain JSON", lambda: f('{"name": "Ana", "days": 7}'), {"name": "Ana", "days": 7}, None),
        ("the example, with a code fence and nesting",
         lambda: f('Sure! Here it is:\n```json\n{"name": "Ana", "tags": {"a": 1}}\n```'), {"name": "Ana", "tags": {"a": 1}},
         "count braces to find the matching }"),
        ("braces inside a string", lambda: f('Result: {"note": "smile :} or {", "ok": true} hope that helps'),
         {"note": "smile :} or {", "ok": True}, "ignore braces inside strings"),
        ("an escaped quote in a string", lambda: f('x {"q": "he said \\"hi}\\"", "n": 1} y'), {"q": 'he said "hi}"', "n": 1}, None),
        ("a false start before the real object", lambda: f('Use the {name} field. {"name": "Lee"}'), {"name": "Lee"},
         "if the first candidate doesn't parse, try the next {"),
        ("a list is not an object", lambda: f("[1, 2, 3]"), Raises(ValueError), None),
        ("no JSON at all", lambda: f("I'm sorry, I can't help with that."), Raises(ValueError), None),
    ])


def check_top_p_filter(f):
    g = torch.Generator().manual_seed(2)
    batch = torch.softmax(torch.randn(3, 12, generator=g, dtype=torch.float64) * 2, dim=-1)

    def reference(probs, p):
        out = torch.zeros_like(probs)
        for r in range(probs.shape[0]):
            order = sorted(range(probs.shape[1]), key=lambda i: (-probs[r, i].item(), i))
            total = 0.0
            for i in order:
                out[r, i] = probs[r, i]
                total += probs[r, i].item()
                if total >= p:
                    break
        return out / out.sum(dim=-1, keepdim=True)

    _scenarios("top_p_filter", [
        ("the example", lambda: f(torch.tensor([0.1, 0.5, 0.3, 0.1]), 0.7), torch.tensor([0.0, 0.625, 0.375, 0.0]), None),
        ("keeps the token that crosses p", lambda: f(torch.tensor([0.5, 0.3, 0.2]), 0.6), torch.tensor([0.625, 0.375, 0.0]),
         "keep while the total of the tokens before this one is below p"),
        ("the top token alone is enough", lambda: f(torch.tensor([0.05, 0.9, 0.05]), 0.5), torch.tensor([0.0, 1.0, 0.0]), None),
        ("a batch of rows", lambda: f(batch, 0.8), reference(batch, 0.8), None),
        ("ties keep the earlier token", lambda: f(torch.tensor([0.25, 0.25, 0.25, 0.25]), 0.4), torch.tensor([0.5, 0.5, 0.0, 0.0]),
         "use a stable sort"),
        ("p = 1 keeps everything", lambda: f(batch, 1.0), batch, None),
        ("exactly reaching p is enough", lambda: f(torch.tensor([0.5, 0.3, 0.2]), 0.5), torch.tensor([1.0, 0.0, 0.0]), None),
        ("p = 0 keeps only the top token", lambda: f(torch.tensor([0.2, 0.5, 0.3]), 0.0), torch.tensor([0.0, 1.0, 0.0]), "always keep the most likely token"),
        ("many ties keep the earliest", lambda: f(torch.full((64,), 1 / 64), 0.1), torch.cat([torch.full((7,), 1 / 7), torch.zeros(57)]), "use a stable sort"),
    ])


def _reference_bm25(query, docs, k1=1.5, b=0.75):
    n = len(docs)
    avgdl = sum(len(d) for d in docs) / n
    out = []
    for d in docs:
        s = 0.0
        for q in query:
            nq = sum(1 for dd in docs if q in dd)
            idf = math.log((n - nq + 0.5) / (nq + 0.5) + 1)
            tf = d.count(q)
            s += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(d) / avgdl))
        out.append(s)
    return np.array(out)


def check_bm25_scores(f):
    docs = [d.lower().split() for d in [
        "loans can be renewed up to three times online or by phone",
        "a lost card can be replaced at any desk for two pounds",
        "printing costs ten pence per page in black and white",
        "renewed loans renewed again renewed renewed",
        "the library card is free card card",
    ]]
    _scenarios("bm25_scores", [
        ("a one-word query", lambda: f(["renewed"], docs), _reference_bm25(["renewed"], docs), None),
        ("a longer query", lambda: f("lost library card".split(), docs), _reference_bm25("lost library card".split(), docs),
         "n(q) counts documents containing the word, not occurrences"),
        ("a repeated query word counts twice", lambda: f(["card", "card"], docs), 2 * _reference_bm25(["card"], docs), None),
        ("a word in no document", lambda: f(["swimming"], docs), np.zeros(5), None),
        ("k1 = 0.5, b = 0.3", lambda: f("renewed loans phone".split(), docs, k1=0.5, b=0.3),
         _reference_bm25("renewed loans phone".split(), docs, 0.5, 0.3), None),
        ("b = 0 ignores document length", lambda: f(["card"], [["card"], ["card"] + ["x"] * 20, ["y"]], b=0.0),
         _reference_bm25(["card"], [["card"], ["card"] + ["x"] * 20, ["y"]], b=0.0), None),
    ])


def check_LoRALinear(cls):
    x = torch.randn(5, 6, generator=torch.Generator().manual_seed(3))
    A_known = torch.arange(12, dtype=torch.float32).reshape(2, 6) / 10
    B_known = torch.arange(8, dtype=torch.float32).reshape(4, 2) / 10 - 0.3

    def base_layer(bias=True):
        torch.manual_seed(0)
        return nn.Linear(6, 4, bias=bias)

    def make(bias=True):
        return cls(base_layer(bias), r=2, alpha=4)

    def expected(bias=True, with_lora=True):
        base = base_layer(bias)
        with torch.no_grad():
            return base(x) + (2.0 * (x @ A_known.T @ B_known.T) if with_lora else 0)

    def with_known(layer):
        with torch.no_grad():
            layer.lora_A.copy_(A_known)
            layer.lora_B.copy_(B_known)
        return layer

    def output(bias=True):
        with torch.no_grad():
            return with_known(make(bias))(x)

    def merged_output(bias=True):
        with torch.no_grad():
            return with_known(make(bias)).merged()(x)

    def grads_only_lora():
        layer = make()
        with torch.no_grad():
            layer.lora_B.fill_(0.1)
        layer(x).sum().backward()
        return [layer.base.weight.grad is None, layer.base.bias.grad is None, layer.lora_A.grad is not None, layer.lora_B.grad is not None]

    def merge_is_separate():
        layer = with_known(make())
        before = layer.base.weight.detach().clone()
        merged = layer.merged()
        return [type(merged).__name__, merged is not layer.base, torch.equal(layer.base.weight, before)]

    def starts():
        with torch.no_grad():
            return make()(x)

    _scenarios("LoRALinear", [
        ("starts out identical to the base layer", starts, expected(with_lora=False), "B starts at zero"),
        ("shapes of A and B", lambda: [tuple(make().lora_A.shape), tuple(make().lora_B.shape)], [(2, 6), (4, 2)], None),
        ("only A and B are trainable", lambda: sum(p.numel() for p in make().parameters() if p.requires_grad), 2 * 6 + 4 * 2,
         "freeze the base layer's weight and bias"),
        ("gradients reach only A and B", grads_only_lora, [True, True, True, True], None),
        ("A is random, not zero", lambda: bool(make().lora_A.abs().sum() > 0), True, "initialise A with kaiming_uniform_"),
        ("output with known A and B", output, expected(), "scale the correction by alpha / r"),
        ("a base layer without bias", lambda: output(False), expected(False), None),
        ("the merged layer gives the same output", merged_output, expected(), "W + scaling * B @ A"),
        ("merging without bias", lambda: merged_output(False), expected(False), None),
        ("merged() returns a new nn.Linear and leaves the base alone", merge_is_separate, ["Linear", True, True], None),
    ])


def check_quantize_int8(f):
    g = torch.Generator().manual_seed(4)
    w = torch.randn(16, 64, generator=g) * torch.linspace(0.01, 3, 16).unsqueeze(1)

    def max_error_ratio():
        q, s = f(w)
        return bool(((q.float() * s - w).abs() <= s / 2 + 1e-6).all())

    def edge():
        row = torch.tensor([[1.0, -1.0, 0.5, 1.0 - 1e-7]])
        q, _ = f(row)
        return q[0].tolist()

    _scenarios("quantize_int8", [
        ("the example", lambda: [f(torch.tensor([[0.5, -1.27, 0.0]]))[0][0].tolist(), f(torch.tensor([[0.5, -1.27, 0.0]]))[1].tolist()],
         [[50, -127, 0], [[0.01]]], None),
        ("dtypes and shapes", lambda: [str(f(w)[0].dtype), tuple(f(w)[0].shape), str(f(w)[1].dtype), tuple(f(w)[1].shape)],
         ["torch.int8", (16, 64), "torch.float32", (16, 1)], "one scale per row, as a column"),
        ("one scale per row", lambda: f(w)[1], w.abs().amax(dim=1, keepdim=True) / 127, "scale = max(|row|) / 127"),
        ("every value within half a step", max_error_ratio, True, None),
        ("the largest value in each row becomes 127 or -127", lambda: f(w)[0].abs().amax(dim=1), torch.full((16,), 127, dtype=torch.int8), None),
        ("rounding to the nearest step", edge, [127, -127, 64, 127], None),
        ("a row of zeros", lambda: [f(torch.zeros(2, 3))[0].tolist(), f(torch.zeros(2, 3))[1].tolist()], [[[0, 0, 0], [0, 0, 0]], [[1.0], [1.0]]], None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `cosine_similarity_matrix`

In [2]:
def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A, B = np.asarray(A, dtype=float), np.asarray(B, dtype=float)
    norm_a = np.linalg.norm(A, axis=1, keepdims=True)
    norm_b = np.linalg.norm(B, axis=1, keepdims=True)
    A_unit = np.divide(A, norm_a, out=np.zeros_like(A), where=norm_a > 0)
    B_unit = np.divide(B, norm_b, out=np.zeros_like(B), where=norm_b > 0)
    return A_unit @ B_unit.T

In [3]:
check_cosine_similarity_matrix(cosine_similarity_matrix)

PASS  all 5 tests passed for cosine_similarity_matrix


Normalising every row to length 1 first turns the whole calculation into one matrix product: for unit vectors, the dot product **is** the cosine similarity. That's exactly what `normalize_embeddings=True` does in the lesson, and why vector databases often store normalised vectors.

`np.divide(..., where=norm > 0)` divides only where the length is positive and leaves zeros elsewhere, so an all-zero row gives similarity 0 instead of `0 / 0 = nan`. A single `nan` is dangerous in a search system: `argsort` puts it in an unpredictable place, and it can silently become the "best" match.

`keepdims=True` keeps the lengths as a column `(n, 1)`, so dividing broadcasts row by row. Without it, an `(n,)` array divides the **columns** instead, which gives wrong numbers, or an error when the shapes don't line up.

### Exercise 2: `top_k_indices`

In [4]:
def top_k_indices(scores, k: int) -> list:
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="stable")
    return [int(i) for i in order[:k]]

In [5]:
check_top_k_indices(top_k_indices)

PASS  all 6 tests passed for top_k_indices


`np.argsort(-scores)` sorts from highest to lowest. The `kind="stable"` part matters for ties: a stable sort keeps equal items in their original order, so the earlier position wins. NumPy's default sort isn't stable, and with ties it can return them in any order, which makes a search system give different results for the same query on different computers.

Sorting `-scores` instead of reversing an ascending sort (`np.argsort(scores)[::-1]`) is what keeps that tie order: reversing would put the **later** of two equal scores first.

For millions of scores, `np.argpartition` finds the top `k` without sorting everything, and you then sort only those `k`. Vector databases go further with approximate indexes, as the lesson mentions.

### Exercise 3: `chunk_words`

In [6]:
def chunk_words(text: str, size: int, overlap: int) -> list:
    if size < 1 or not 0 <= overlap < size:
        raise ValueError("need size >= 1 and 0 <= overlap < size")
    words = text.split()
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks

In [7]:
check_chunk_words(chunk_words)

PASS  all 9 tests passed for chunk_words


The overlap is there so that a sentence cut at the end of one chunk appears whole at the start of the next. Without it, a fact split across two chunks might not be found by either. Typical settings are chunks of a few hundred words with 10 to 20% overlap.

The `break` stops as soon as a chunk reaches the end. Without it, the loop keeps going and adds chunks such as `"g"` that are entirely contained in the previous one: duplicates in the index, which waste space and can fill the retrieved results with the same text twice. One check catches exactly that.

Why must the overlap be smaller than the size? With `overlap == size` the step is 0 and the loop never moves forward; `range` with a step of 0 raises an error, and a `while` loop would run forever.

Real systems usually count **tokens** rather than words (the embedding model and the language model both have token limits), and prefer to cut at sentence or paragraph boundaries. The structure of the code stays the same.

### Exercise 4: `format_chatml`

In [8]:
def format_chatml(messages, add_generation_prompt: bool = False) -> str:
    parts = []
    for message in messages:
        if message["role"] not in ("system", "user", "assistant"):
            raise ValueError(f"unknown role {message['role']!r}")
        parts.append(f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n")
    if add_generation_prompt:
        parts.append("<|im_start|>assistant\n")
    return "".join(parts)

In [9]:
check_format_chatml(format_chatml)

PASS  all 5 tests passed for format_chatml


This is what `tokenizer.apply_chat_template(..., tokenize=False)` produces for ChatML models, written out by hand. The real SmolLM2 template does one extra thing: if there's no system message, it adds a default one ("You are a helpful AI assistant named SmolLM..."), which is why the lesson's prompts show it.

Writing the format yourself is useful for understanding, but in real code use `apply_chat_template`: every model family has its own format (Llama, Mistral and Gemma all differ), and a prompt in the wrong format can make a good model give poor answers, without any error to warn you.

`<|im_start|>` and `<|im_end|>` are **special tokens**: each is a single token in the vocabulary, not the eleven characters you see. When the model generates `<|im_end|>`, generation stops; that's how the model ends its turn.

Checking the roles catches a common bug, a typo like `"assistent"`, which the model would read as just another word and which quietly breaks the conversation.

## Medium

### Exercise 5: `build_rag_prompt`

In [10]:
def build_rag_prompt(question: str, chunks, max_chars=None) -> str:
    lines = []
    for n, chunk in enumerate(chunks, start=1):
        line = f"[{n}] {chunk}"
        candidate = "\n".join(lines + [line])
        if max_chars is not None and len(candidate) > max_chars:
            break
        lines.append(line)
    sources = "\n".join(lines) if lines else "(no sources)"
    return ("Answer the question using only the sources below. "
            "If they don't contain the answer, say \"I don't know\".\n\n"
            f"Sources:\n{sources}\n\nQuestion: {question}\nAnswer:")

In [11]:
check_build_rag_prompt(build_rag_prompt)

PASS  all 6 tests passed for build_rag_prompt


Every language model has a **context window**, a maximum number of tokens it can read, and longer prompts are slower and cost more. So a RAG system needs a budget for the retrieved text. Since chunks arrive best first, adding them in order and stopping at the first that doesn't fit keeps the most relevant ones. (Characters stand in for tokens here; a real system would count tokens with the model's tokenizer.)

Stopping at the first chunk that doesn't fit, rather than skipping it and trying smaller ones further down, keeps the sources in rank order and keeps the numbering simple. Both choices are defensible; what matters is that it's deliberate and tested.

Numbering the sources lets the model cite them ("[2]"), and lets your program check those citations: a reply that cites `[5]` when only three sources were given is a sign something went wrong.

"(no sources)" is better than an empty block: it tells the model plainly that there's nothing to go on, which makes "I don't know" the natural answer.

### Exercise 6: `retrieval_metrics`

In [12]:
def retrieval_metrics(results, relevant, k: int) -> dict:
    if len(results) != len(relevant):
        raise ValueError("results and relevant must have one entry per query")
    recalls, reciprocal_ranks = [], []
    for ranked, rel in zip(results, relevant):
        found = sum(1 for doc in ranked[:k] if doc in rel)
        recalls.append(found / len(rel))
        rr = 0.0
        for position, doc in enumerate(ranked, start=1):
            if doc in rel:
                rr = 1 / position
                break
        reciprocal_ranks.append(rr)
    return {"recall@k": float(np.mean(recalls)), "mrr": float(np.mean(reciprocal_ranks))}

In [13]:
check_retrieval_metrics(retrieval_metrics)

PASS  all 6 tests passed for retrieval_metrics


The two numbers answer different questions:

- **recall@k**: does the right material make it into the prompt at all? For RAG that's the one that matters most, since the model can only use what it's given. `k` should match how many chunks you actually put in the prompt.
- **MRR** (mean reciprocal rank): how high up is the first right answer? A system that always puts it first scores 1; one that always puts it third scores 0.33. It rewards good ordering, which matters when you can only afford a few chunks.

Details the checks look at: recall divides by the number of **relevant** documents (not by `k`), so a query with two relevant documents and one found scores 0.5; the reciprocal rank uses the first relevant document's position even if it's beyond `k`; and positions count from 1, so finding it first gives 1, not `1 / 0`.

These numbers only mean something with a good test set: realistic questions, labelled by someone who knows which documents answer them. Building that set is usually the slowest part of improving a search system, and the most valuable.

### Exercise 7: `answer_f1`

In [14]:
def answer_f1(prediction: str, truth: str) -> float:
    def normalise(text):
        text = text.lower()
        text = "".join(ch for ch in text if ch not in string.punctuation)
        return [w for w in text.split() if w not in ("a", "an", "the")]

    pred, gold = normalise(prediction), normalise(truth)
    if not pred and not gold:
        return 1.0
    common = sum((collections.Counter(pred) & collections.Counter(gold)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(gold)
    return 2 * precision * recall / (precision + recall)

In [15]:
check_answer_f1(answer_f1)

PASS  all 8 tests passed for answer_f1


`Counter(a) & Counter(b)` keeps each word with the **smaller** of its two counts, which is exactly "words in common, with repeats". Using sets instead would treat "no no no" and "no" as identical, and lists with `in` would count every repeat in the prediction as a match.

Normalising first means "The library card costs £2." and "library card costs £2" aren't penalised for "The" and the full stop. Removing articles is a convention from the SQuAD benchmark; it's a reasonable choice for English, but every choice in a metric changes the numbers, so always report how answers were normalised.

F1 gives **partial credit**, which suits language models: their answers are usually longer than the reference ("It costs 2 pounds" against "2 pounds"), and exact match would score that 0. But F1 is still about words, not meaning: "2 pounds" and "two pounds" share only one word, and "not allowed" against "allowed" scores 0.67 despite meaning the opposite. That's why the lesson recommends reading answers, and why longer answers are usually graded by people or, with care, by another model.

### Exercise 8: `extract_json`

In [16]:
def extract_json(text: str) -> dict:
    try:
        data = json.loads(text.strip())
        if isinstance(data, dict):
            return data
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    while start != -1:
        depth, in_string, escaped = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_string:
                if escaped:
                    escaped = False
                elif ch == "\\":
                    escaped = True
                elif ch == '"':
                    in_string = False
            elif ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        data = json.loads(text[start:i + 1])
                        if isinstance(data, dict):
                            return data
                    except json.JSONDecodeError:
                        pass
                    break
        start = text.find("{", start + 1)
    raise ValueError("no JSON object found")

In [17]:
check_extract_json(extract_json)

PASS  all 7 tests passed for extract_json


Counting braces is how you find where an object ends: `{` adds one, `}` takes one away, and the object closes when the count gets back to zero. The string handling is what makes it dependable: a value like `"smile :}"` contains a brace that isn't part of the structure, so braces inside strings must be ignored, and inside strings `\"` is an escaped quote, not the end of the string.

If the first candidate doesn't parse (models sometimes write things like `{name}` in a sentence before the real JSON), the loop moves on to the next `{`. One check has exactly that.

Trying `json.loads` on the whole text first handles the easy case quickly, and the `isinstance(data, dict)` checks make sure a reply that's just a number or a list isn't accepted as an object.

This is about as far as parsing should go. If a reply still doesn't contain valid JSON, the right response is to retry or fall back, as in the lesson, not to guess at what the model meant.

### Exercise 9: `top_p_filter`

In [18]:
def top_p_filter(probs: torch.Tensor, p: float) -> torch.Tensor:
    sorted_probs, order = torch.sort(probs, dim=-1, descending=True, stable=True)
    cumulative = sorted_probs.cumsum(dim=-1)
    keep_sorted = (cumulative - sorted_probs) < p          # keep while the total *before* this token is below p
    keep_sorted[..., 0] = True
    keep = torch.zeros_like(keep_sorted).scatter(-1, order, keep_sorted)
    filtered = torch.where(keep, probs, torch.zeros_like(probs))
    return filtered / filtered.sum(dim=-1, keepdim=True)

In [19]:
check_top_p_filter(top_p_filter)

PASS  all 9 tests passed for top_p_filter


The trick is `cumulative - sorted_probs`: the total probability of the tokens **before** each one. A token is kept if that total is still below `p`, which keeps exactly the tokens needed to reach `p`, including the one that crosses it. Comparing `cumulative < p` instead drops that crossing token, so the kept set adds up to **less** than `p`, and when the top token alone has more than `p` it would keep nothing at all.

`scatter` puts the keep/drop decisions back in the original token order: `order[j]` is the original position of the `j`-th most likely token.

Top-p adapts to the model's confidence, which is why it's popular. When the model is sure, one or two tokens make up 90% and only they are kept; when it's unsure, many tokens are needed to reach 90% and the choice stays wide. Top-k (Module 19) keeps a fixed number whatever the confidence. The two are often combined, with a temperature applied first.

## Hard

### Exercise 10: `bm25_scores`

In [20]:
def bm25_scores(query, documents, k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    n_docs = len(documents)
    lengths = np.array([len(d) for d in documents], dtype=float)
    avgdl = lengths.mean()
    counts = [collections.Counter(d) for d in documents]
    containing = collections.Counter(word for c in counts for word in c)
    scores = np.zeros(n_docs)
    for q in query:
        idf = math.log((n_docs - containing[q] + 0.5) / (containing[q] + 0.5) + 1)
        f = np.array([c[q] for c in counts], dtype=float)
        scores += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * lengths / avgdl))
    return scores

In [21]:
check_bm25_scores(bm25_scores)

PASS  all 6 tests passed for bm25_scores


BM25 improves on plain word counting in three ways, one per part of the formula:

- **Rare words count more** (`idf`). "renew" in a library handbook is far more informative than "the", which appears everywhere and gets an idf near 0. The `+ 1` inside the logarithm keeps idf positive even for words in more than half the documents.
- **Repeats have diminishing returns** (`k1`). The term `f (k1 + 1) / (f + k1 ...)` grows with the count but levels off: the tenth mention of a word adds much less than the first. With `k1 = 0` only presence matters.
- **Long documents are handicapped** (`b`). A long document contains more words by chance, so its counts are scaled down relative to the average length. `b = 0` turns this off; `b = 1` applies it fully.

Counting which documents contain each word with `Counter(word for c in counts for word in c)` works because each document's `Counter` lists each word once, so the outer count is "number of documents", not "number of occurrences".

BM25 is decades old (its name means "Best Match 25") and still a strong baseline. It's very good at exact names, numbers and rare terms, which is why the lesson suggests **hybrid search**: combine BM25 scores with embedding similarity, so each covers the other's weak spots.

### Exercise 11: `LoRALinear`

In [22]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + self.scaling * (x @ self.lora_A.T @ self.lora_B.T)

    def merged(self) -> nn.Linear:
        layer = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        with torch.no_grad():
            layer.weight.copy_(self.base.weight + self.scaling * self.lora_B @ self.lora_A)
            if self.base.bias is not None:
                layer.bias.copy_(self.base.bias)
        return layer

In [23]:
check_LoRALinear(LoRALinear)

PASS  all 10 tests passed for LoRALinear


Why `B` starts at zero and `A` doesn't: with `B = 0` the correction `B A` is exactly zero, so the wrapped layer starts out identical to the original, and fine-tuning begins from the pretrained model rather than a damaged copy of it. If **both** were zero, the gradients of `A` and `B` would both be zero too (each one's gradient is multiplied by the other), and nothing would ever train. Starting `A` random and `B` at zero avoids both problems, and is what the `peft` library does.

`x @ A.T @ B.T` first squeezes `x` down to `r` numbers, then expands back to `out_features`. That costs about `r * (in + out)` multiplications instead of `in * out` for a full extra matrix, and it's why LoRA is cheap in memory: only `A` and `B` need gradients and optimiser state.

`scaling = alpha / r` keeps the size of the correction roughly the same when you change `r`, so a learning rate that works at rank 8 still works at rank 16.

`merged()` shows LoRA's other advantage: after training, `W + scaling * B A` is just a matrix of the original shape, so the fine-tuned model runs exactly as fast as the original. The check confirms that the merged layer gives the same outputs, and that it's a separate object, so merging doesn't change the original layer.

### Exercise 12: `quantize_int8`

In [24]:
def quantize_int8(weight: torch.Tensor):
    weight = weight.float()
    scales = weight.abs().amax(dim=1, keepdim=True) / 127
    scales = torch.where(scales == 0, torch.ones_like(scales), scales)
    q = torch.round(weight / scales).clamp(-127, 127).to(torch.int8)
    return q, scales

In [25]:
check_quantize_int8(quantize_int8)

PASS  all 7 tests passed for quantize_int8


Each number now takes 1 byte instead of 4, plus one scale per row, so a large matrix shrinks almost 4 times. The cost is precision: within a row, every weight is rounded to one of 255 evenly spaced values, so the error is at most half a step, `scale / 2`.

**Per row** matters. With one scale for the whole matrix, a single large weight anywhere would set the step size for every row, and rows of small weights would be rounded to almost nothing. Real quantisation schemes go further, with a scale for every block of 32 to 128 weights, and treat the rare very large weights ("outliers") specially, which is how 4-bit models keep most of their quality.

**Symmetric** means zero is always represented exactly and the range is the same on both sides. Using -127 to 127 (not -128) keeps it symmetric.

`.clamp` is a cheap safety net. With the scale computed from the row itself, the values already fall between -127 and 127, but quantisation code often reuses scales computed elsewhere (for example on a sample of data), and converting an out-of-range float to `int8` doesn't raise an error: it gives a wrong number, sometimes with the opposite sign. Clamping guarantees the range whatever happens upstream.

The zero-row rule avoids `0 / 0`. Rows of zeros are rare in trained models, but common in freshly created ones, and a quantiser that produces `nan` for them would break silently.

## What's next

The mini project in `04_mini_project.ipynb` puts these pieces together: a RAG assistant that answers questions about this course from its own README files. You'll compare keyword, BM25 and embedding search with recall@k and MRR, generate answers with sources, and measure how often they're right.